In [33]:
import pandas as pd

In [34]:
df = pd.read_csv("C:/Users/amank/Downloads/archive (3)/online_retail.csv")

In [35]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,01-12-2010 08:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,01-12-2010 08:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,01-12-2010 08:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,01-12-2010 08:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,01-12-2010 08:26,3.39,17850.0,United Kingdom


In [36]:
print("size of data withot clean ", df.shape)

size of data withot clean  (541909, 8)


In [37]:
print("Missing values:\n", df.isnull().sum())

Missing values:
 InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


In [38]:
df_clean = df.dropna(subset=['CustomerID'])

In [39]:
df_clean = df_clean[(df_clean['Quantity']>0) & (df_clean['UnitPrice']>0)]

In [40]:
print ("After cleaning data: ",df_clean.shape)

After cleaning data:  (397884, 8)


In [41]:
import datetime as dt

In [42]:
df_clean['TotalAmount'] = df_clean['Quantity']* df_clean['UnitPrice']

In [43]:
# conver data in proper format 
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'], dayfirst=True)

# Latest date 
latest_date = df_clean['InvoiceDate'].max() + dt.timedelta(days=1)

# RFM table
rfm = df_clean.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (latest_date - x.max()).days,
    'InvoiceNo': 'count',
    'TotalAmount': 'sum'
})

# name of column
rfm.rename(columns={'InvoiceDate': 'Recency', 
                    'InvoiceNo': 'Frequency', 
                    'TotalAmount': 'Monetary'}, inplace=True)

# 5. Result print 
print(rfm.head())

            Recency  Frequency  Monetary
CustomerID                              
12346.0         326          1  77183.60
12347.0           2        182   4310.00
12348.0          75         31   1797.24
12349.0          19         73   1757.55
12350.0         310         17    334.40


In [44]:
# R, F, and M score 
rfm['R_Score'] = pd.qcut(rfm['Recency'], q=4, labels=[4, 3, 2, 1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=4, labels=[1, 2, 3, 4])
rfm['M_Score'] = pd.qcut(rfm['Monetary'], q=4, labels=[1, 2, 3, 4])

# add all three score 'RFM_Segment' 
rfm['RFM_Segment'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

# 3. 'Total_Score'  (R, F, M add mathematical  4+4+4 = 12)
rfm['Total_Score'] = rfm[['R_Score', 'F_Score', 'M_Score']].sum(axis=1)

# 4. Output check karna
print(rfm.head())

            Recency  Frequency  Monetary R_Score F_Score M_Score RFM_Segment  \
CustomerID                                                                     
12346.0         326          1  77183.60       1       1       4         114   
12347.0           2        182   4310.00       4       4       4         444   
12348.0          75         31   1797.24       2       2       4         224   
12349.0          19         73   1757.55       3       3       4         334   
12350.0         310         17    334.40       1       1       2         112   

            Total_Score  
CustomerID               
12346.0               6  
12347.0              12  
12348.0               8  
12349.0              10  
12350.0               4  


In [45]:
def segment_customer(df):
    if df['Total_Score'] >= 11:
        return 'Champions'
    elif df['Total_Score'] >= 9:
        return 'Loyal Customers'
    elif df['Total_Score'] >= 7:
        return 'Potential Loyalists'
    elif df['Total_Score'] >= 5:
        return 'Needs Attention'
    else:
        return 'Lost / At Risk'


rfm['Customer_Profile'] = rfm.apply(segment_customer, axis=1)


print("Customer Segments ki Summary:\n")
print(rfm['Customer_Profile'].value_counts())

Customer Segments ki Summary:

Customer_Profile
Needs Attention        980
Potential Loyalists    914
Loyal Customers        846
Champions              839
Lost / At Risk         759
Name: count, dtype: int64
